### Binary Classification Metrics

Binary classification predicts one of two possible classes, such as **positive / negative**, **disease / no disease**, or **spam / not spam**.

This notebook explains the classification metrics used to evaluate binary classifiers:
- Confusion Matrix
- Accuracy
- Precision
- Recall (Sensitivity)
- Specificity
- F1 Score
- Classification Report

We use Scikit-learn's **Breast Cancer Wisconsin** dataset. It contains two classes: malignant and benign. The metrics are calculated on a held-out test set.

### Load and Prepare the Dataset

The Breast Cancer Wisconsin dataset is built into Scikit-learn, so no separate download is required.

The target labels are:
- `0`: malignant
- `1`: benign

For metric interpretation, we will explicitly treat **malignant (0) as the positive class** in the binary examples. This is an important choice: precision and recall depend on which class is considered positive.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)

In [2]:
dataset = load_breast_cancer()

X = pd.DataFrame(dataset.data, columns=dataset.feature_names)

# Convert target so malignant is the positive class (1), benign is negative (0)
y = (dataset.target == 0).astype(int)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Positive class: malignant (1)")
print("Negative class: benign (0)")
print("Test accuracy:", accuracy_score(y_test, y_pred))

Positive class: malignant (1)
Negative class: benign (0)
Test accuracy: 0.9649122807017544


### 1. Accuracy

Accuracy measures the proportion of all predictions that are correct.

$$
Accuracy = \\frac{TP + TN}{TP + TN + FP + FN}
$$

**Example:** If a model makes 100 predictions and 90 are correct, its accuracy is $90/100 = 0.90$ (90%).

**When to use**
- When classes are reasonably balanced.
- When false positives and false negatives have similar costs.
- As a simple overall performance measure.

**Advantages**
- Easy to understand and communicate.
- Uses all four confusion-matrix outcomes.

**Disadvantages**
- Can be misleading on imbalanced datasets.
- Does not show which type of error the model makes.

In [4]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f} ({accuracy:.1%})")

Accuracy: 0.965 (96.5%)


### 2. Confusion Matrix

A confusion matrix compares the actual labels with the model's predictions.

For binary classification, the matrix is arranged as:

| | Predicted Positive (1)| Predicted Negative (0)|
|:--|--:|--:|
| **Actual Positive (1)** | True Positive (TP) | False Negative (FN) |
| **Actual Negative(0)** | False Positive (FP) | True Negative (TN) |

- **TP :** The actual class is positive and the model predicts positive.
- **TN :** The actual class is negative and the model predicts negative.
- **FP :** The actual class is negative but the model predicts positive (false alarm).
- **FN :** The actual class is positive but the model predicts negative (missed positive case).

For this dataset, a false negative means a malignant case was predicted as benign. This can be a particularly serious error in a medical screening context.

In [5]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["Actual: Benign (0)", "Actual: Malignant (1)"],
    columns=["Predicted: Benign (0)", "Predicted: Malignant (1)"]
)
cm_df

,Predicted: Benign (0),Predicted: Malignant (1)
Actual: Benign (0),71,1
Actual: Malignant (1),3,39


### 3 Precision

Precision answers : **Of all observations predicted as positive, how many were actually positive?**

$$
Precision = \\frac{TP}{TP + FP}
$$

High precision means the model produces relatively few false positives.

**When to use**
- When false positives are costly.
- For example, when a positive prediction triggers an expensive or invasive follow-up test.

**Advantages**
- Focuses on the reliability of positive predictions.
- Useful when false alarms matter.

**Disadvantages**
- Does not measure how many actual positive cases were missed.
- Can be high even when recall is low.

In [6]:
precision = precision_score(y_test, y_pred, pos_label=1)
print(f"Precision (malignant): {precision:.3f}")

Precision (malignant): 0.975


### 4. Recall (Sensitivity)

Recall measures : **Of all actual positive observations, how many did the model correctly identify?**

$$
Recall = \\frac{TP}{TP + FN}
$$

High recall means the model misses fewer positive cases.

**When to use**
- When false negatives are costly.
- For example, in an initial medical screening task where missing a disease case is dangerous.

**Advantages**
- Measures how effectively the model finds positive cases.
- Particularly useful when missing a positive case is costly.

**Disadvantages**
- Does not account for how many false positives are produced.
- A model can achieve high recall by predicting many observations as positive.

In [7]:
recall = recall_score(y_test, y_pred, pos_label=1)
print(f"Recall / Sensitivity (malignant): {recall:.3f}")

Recall / Sensitivity (malignant): 0.929


### 5. Specificity

Specificity measures: **Of all actual negative observations, how many did the model correctly identify?**

$$
Specificity = \\frac{TN}{TN + FP}
$$

**When to use**
- When correctly ruling out negative cases matters.
- When false positives create unnecessary follow-up work or costs.

**Advantages**
- Measures performance on the negative class.
- Complements recall, which focuses on the positive class.

**Disadvantages**
- Does not measure how many positive cases are detected.
- Should be considered alongside recall, not alone.

In [8]:
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) else 0
print(f"Specificity: {specificity:.3f}")

Specificity: 0.986


### 6. F1 Score

F1 Score is the harmonic mean of precision and recall. It balances both metrics and becomes low if either precision or recall is low.

$$
F1 = 2 \\times \\frac{Precision \\times Recall}{Precision + Recall}
$$

**When to use**
- When you need a balance between precision and recall.
- When the positive class is important and the dataset may be imbalanced.

**Advantages**
- Combines precision and recall into one score.
- More informative than accuracy in many imbalanced classification tasks.

**Disadvantages**
- Does not include true negatives directly.
- Hides the trade-off between precision and recall; always inspect both as well.

In [9]:
f1 = f1_score(y_test, y_pred, pos_label=1)
print(f"F1 Score (malignant): {f1:.3f}")

F1 Score (malignant): 0.951


### Classification Report

Scikit-learn's `classification_report` summarizes precision, recall, F1 score, and support for each class.

- **Support :** Number of actual observations belonging to a class.
- **Macro average :** Calculates the metric for each class, then takes the unweighted mean. Every class has equal influence.
- **Weighted average :** Calculates the metric for each class, then averages using each class's support. Larger classes have more influence.

The report also includes accuracy. For binary classification, inspect the row for the positive class as well as the averages.

In [10]:
print(classification_report(
    y_test, y_pred,
    target_names=["Benign (negative)", "Malignant (positive)"],
    zero_division=0
))

                      precision    recall  f1-score   support

   Benign (negative)       0.96      0.99      0.97        72
Malignant (positive)       0.97      0.93      0.95        42

            accuracy                           0.96       114
           macro avg       0.97      0.96      0.96       114
        weighted avg       0.97      0.96      0.96       114



### Which Metric Should You Choose?

| Goal / situation | Metric to focus on | Reason |
|:--|:--|:--|
| Classes balanced and error costs similar | Accuracy | Simple overall correctness |
| False positives are expensive | Precision | Limits false alarms among positive predictions |
| False negatives are expensive | Recall / Sensitivity | Finds more actual positive cases |
| Correctly identifying negatives is important | Specificity | Measures true negative rate |
| Need a balance of precision and recall | F1 Score | Combines both in one measure |
| Need a complete class-wise summary | Classification Report | Shows precision, recall, F1, and support |

There is no universal rule such as “90% accuracy is always good.” A good score depends on class balance, the cost of different errors, the application, and the performance of a suitable baseline.

### Key Takeaways

- Start with the confusion matrix to understand the kinds of mistakes.
- Accuracy alone can hide poor performance on an important class.
- Precision focuses on false positives; recall focuses on false negatives.
- Specificity measures how well the model recognizes the negative class.
- F1 balances precision and recall, but does not replace inspecting them separately.
- Choose the positive class deliberately and report it clearly.
- In real applications, choose thresholds and metrics based on the costs of errors, and validate decisions using training/validation data before final test evaluation.